# 🗄️ Clase 2: Bases de Datos y Limpieza de Datos

**Curso: Modelos de AI en Negocios (MAIN) - IA Aplicada a la Industria**
Universidad de Santiago de Chile · Departamento de Ingeniería Industrial

---

## 🏭 Contexto del caso

Somos analistas de datos en una empresa industrial. El equipo de calidad nos ha entregado un dataset real extraído de **Kaggle** con **1.000 registros de defectos detectados en procesos de manufactura**, incluyendo el tipo de defecto, su severidad, la fecha de detección, la ubicación en el producto, el método de inspección utilizado y el costo de reparación.

**El problema:** los datos tienen errores — valores faltantes, fechas guardadas como texto y costos imposibles — que impiden usarlos directamente para tomar decisiones.

**Nuestro objetivo:**
1. Detectar y corregir los problemas de calidad del dataset
2. Calcular el **costo promedio de reparación** por tipo de defecto y severidad
3. Identificar **qué tipo de defecto es más costoso** y **qué método de inspección lo detecta mejor**
4. Exportar un CSV limpio y gráficos listos para presentar a gerencia

---

## 📋 Diccionario de datos (columnas del CSV)

| Columna | Descripción | Tipo esperado |
|---|---|---|
| `defect_id` | Identificador único del defecto | Entero |
| `product_id` | Identificador del producto afectado | Entero |
| `defect_type` | Tipo de defecto: Structural, Functional, Cosmetic | Texto categórico |
| `defect_date` | Fecha de detección del defecto | Fecha (guardada como texto — dato sucio) |
| `defect_location` | Ubicación en el producto: Surface, Component, Internal | Texto categórico |
| `severity` | Severidad: Minor, Moderate, Critical | Texto / **NaN** (dato sucio) |
| `inspection_method` | Método de inspección: Visual, Automated, Manual | Texto / **NaN** (dato sucio) |
| `repair_cost` | Costo de reparación en moneda local | Float (algunos valores **negativos** — error de entrada) |

---

## ⚠️ Problemas conocidos en los datos (lo que vamos a solucionar)

- **Valores nulos** en `severity` (~10%) y `inspection_method` (~8%) — campos no siempre registrados
- **Fechas como texto** en `defect_date` en lugar de tipo fecha/datetime
- **Costos negativos** en `repair_cost` (~3%) — errores de ingreso de datos

> 💡 **Nota sobre el dataset:** El CSV fue descargado de [Kaggle — Manufacturing Defects](https://www.kaggle.com/datasets/fahmidachowdhury/manufacturing-defects). Es un dataset sintético pero realista, diseñado para análisis de control de calidad. Los problemas de calidad de datos fueron **inyectados intencionalmente** en el Paso 0 para simular condiciones reales de manufactura.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELDA 1: IMPORTAR LIBRERÍAS
# ════════════════════════════════════════════════════════════════════════
# Una "librería" es un conjunto de herramientas que alguien más programó
# y que nosotros podemos reutilizar sin tener que escribirlo desde cero.
# Es como descargar una app: la instalas una vez y ya puedes usarla.
#
# La sintaxis es:   import NOMBRE_LIBRERIA as APODO
# El "apodo" (alias) nos permite escribir menos código. Por ejemplo:
#   en vez de escribir "pandas.read_csv()" → escribimos solo "pd.read_csv()"
# ════════════════════════════════════════════════════════════════════════

import numpy as np
# numpy (abreviado "np") → trabaja con listas y matrices de números de forma eficiente.
# Lo usaremos para generar posiciones aleatorias dentro del dataset.

import pandas as pd
# pandas (abreviado "pd") → la librería más importante de este laboratorio.
# Nos permite leer archivos CSV y trabajar con DataFrames (tablas de datos).
# Un DataFrame es como una hoja de Excel pero dentro de Python.

import matplotlib.pyplot as plt
# matplotlib.pyplot (abreviado "plt") → herramienta para crear gráficos
# (barras, líneas, histogramas, etc.)

import matplotlib.ticker as mticker
# mticker → herramientas auxiliares para formatear los ejes de los gráficos
# (por ejemplo, mostrar porcentajes o fechas en los ejes)

# La función print() muestra texto en pantalla.
# Nos sirve para confirmar que las librerías se cargaron sin errores.
print('✅ Librerías cargadas correctamente:')
print('  - numpy      : operaciones numéricas y selección aleatoria de posiciones')
print('  - pandas     : leer y limpiar tablas de datos (DataFrames)')
print('  - matplotlib : crear gráficos de barras, líneas y tendencias')

---
### 🗂️ Concepto clave: ¿Qué es un DataFrame?

Un **DataFrame** es la estructura principal de pandas. Piénsalo como **una tabla de Excel dentro de Python**:

| **Concepto en Excel** | **Equivalente en Python / pandas** |
|---|---|
| Hoja de cálculo | `DataFrame` (lo llamamos `df` por convención) |
| Fila | registro / fila |
| Columna | columna / campo |
| Celda específica | `df.loc[número_de_fila, 'nombre_columna']` |
| Abrir un archivo `.csv` | `pd.read_csv('archivo.csv')` |
| Guardar como `.csv` | `df.to_csv('archivo.csv', index=False)` |
| Filtrar filas | `df[df['columna'] > valor]` |
| Columna en particular | `df['nombre_columna']` |

---

### 📌 Notación `df.` — ¿qué significa el punto?

El **punto** (`.`) en Python significa **"aplica esto AL objeto de la izquierda"**.

```python
df.head()         # llama al método head() sobre el DataFrame df → muestra las primeras 5 filas
df.shape          # accede a la propiedad shape del df → devuelve (filas, columnas)
df['severity']    # accede a la columna 'severity' del df → una sola columna
df.isnull()       # aplica isnull() al df → crea una tabla de True/False (True = celda vacía)
```

> 💡 **Regla práctica:** siempre que veas `df.algo()` o `df['algo']`, se está accediendo o aplicando algo sobre la tabla de datos.

---
## 📂 Paso 0: Cargar los datos e inyectar problemas de calidad

El archivo `defects_data.csv` fue descargado desde Kaggle. Para obtenerlo:
- **Opción A (Colab):** Subir el archivo manualmente a la sesión y leerlo con `pd.read_csv('defects_data.csv')`
- **Opción B (con kagglehub):** Ejecutar `kagglehub.dataset_download("fahmidachowdhury/manufacturing-defects")`

Una vez cargado el CSV limpio, **inyectamos nulos y errores** intencionalmente para simular los problemas que encontraríamos en datos reales de manufactura.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PASO 0A: CARGAR EL ARCHIVO CSV
# ════════════════════════════════════════════════════════════════════════

# pd.read_csv('nombre_del_archivo.csv') lee un archivo de texto separado por comas
# y lo convierte automáticamente en un DataFrame (tabla de datos de pandas).
#
# Lo que sucede internamente: pandas abre el archivo, lee la primera fila como
# nombres de columnas, y convierte cada fila siguiente en un registro de la tabla.
#
# El resultado se guarda en la variable "df".
# "df" es el nombre que nosotros elegimos — podría llamarse "tabla" o "datos",
# pero "df" (de DataFrame) es la convención estándar en toda la industria.
df = pd.read_csv('defects_data.csv')

# df.shape es una PROPIEDAD (no función) que devuelve una tupla con 2 valores:
#   df.shape[0] = número de filas    (el [0] accede al primer elemento)
#   df.shape[1] = número de columnas (el [1] accede al segundo elemento)
# Los f-strings (f'...{variable}...') permiten mezclar texto y variables en un print()
print(f'Dataset original cargado: {df.shape[0]} filas, {df.shape[1]} columnas')

# df.columns devuelve los nombres de todas las columnas del DataFrame (como una lista especial)
# list(...) la convierte en una lista simple de Python, más fácil de leer
print(f'Columnas: {list(df.columns)}')

# df.isnull()        → crea una tabla nueva donde True = celda vacía,  False = celda con dato
# .sum()             → suma cada columna (True cuenta como 1, False como 0) = nulos por columna
# .sum() (otra vez)  → suma esos totales → da el total de celdas vacías en todo el dataset
print(f'Nulos originales: {df.isnull().sum().sum()} (el CSV de Kaggle viene limpio)')

# ════════════════════════════════════════════════════════════════════════
# PASO 0B: INYECTAR PROBLEMAS DE CALIDAD (simulación de datos reales)
# ════════════════════════════════════════════════════════════════════════
# En la industria real, estos tipos de errores aparecen solos por fallas
# en el sistema de registro o por errores humanos al ingresar datos.
# Los creamos nosotros aquí para poder practicar cómo detectarlos y corregirlos.

# np.random.seed(42) fija la "semilla" del generador de números aleatorios.
# Sin esta instrucción, cada ejecución generaría distintas posiciones aleatorias.
# Con la semilla fija, todos los estudiantes obtenemos exactamente los mismos resultados.
# El número 42 es arbitrario — podría ser cualquier entero positivo.
np.random.seed(42)

# len(df) devuelve el número de filas total — equivalente a df.shape[0]
# Guardamos el resultado en la variable "n" para usarla varias veces a continuación
n = len(df)

# ────────────────────────────────────────────────────────────────────────
# PROBLEMA 1: Crear valores nulos en la columna 'severity'
# Simulamos que el operador no siempre registra la severidad (~10% de los casos)
# ────────────────────────────────────────────────────────────────────────

# np.random.choice(n, size=X, replace=False) elige X números enteros al azar
# del rango [0, n-1], sin repetir. Son los ÍNDICES (números de fila) que vamos a vaciar.
# int(n * 0.10) calcula el 10% del total de filas como número entero
idx_null_sev = np.random.choice(n, size=int(n * 0.10), replace=False)

# df.loc[lista_de_índices, 'nombre_columna'] accede a celdas específicas del DataFrame.
# Equivale a seleccionar ciertas filas y una columna particular.
# np.nan es el valor especial de Python/pandas que representa "celda vacía" (ausente).
# Al asignarlo, esas celdas quedan en blanco — igual que borrar el contenido en Excel.
df.loc[idx_null_sev, 'severity'] = np.nan

# ────────────────────────────────────────────────────────────────────────
# PROBLEMA 2: Crear valores nulos en 'inspection_method'
# Simulamos que el método de inspección no siempre se registra (~8%)
# ────────────────────────────────────────────────────────────────────────
idx_null_insp = np.random.choice(n, size=int(n * 0.08), replace=False)
df.loc[idx_null_insp, 'inspection_method'] = np.nan

# ────────────────────────────────────────────────────────────────────────
# PROBLEMA 3: Crear costos de reparación negativos (~3%)
# Simulamos un error de signo al ingresar el costo (ej: -350 en vez de 350)
# ────────────────────────────────────────────────────────────────────────
idx_neg_cost = np.random.choice(n, size=int(n * 0.03), replace=False)
# -df.loc[...] multiplica esos valores por -1:
#   un costo de $312.50 → se vuelve -$312.50 (imposible en la realidad)
df.loc[idx_neg_cost, 'repair_cost'] = -df.loc[idx_neg_cost, 'repair_cost']

# Verificamos que los problemas se crearon correctamente
print('\n⚠️  Problemas inyectados para el ejercicio:')
# df["severity"].isnull() → serie True/False, True en cada celda vacía de la columna
# .sum() cuenta cuántos True hay → cuántos nulos tiene esa columna
print(f'   - Nulos en "severity":          {df["severity"].isnull().sum()} registros')
print(f'   - Nulos en "inspection_method": {df["inspection_method"].isnull().sum()} registros')
# (df["repair_cost"] < 0) → True en cada fila donde el costo es negativo
print(f'   - Costos negativos:             {(df["repair_cost"] < 0).sum()} registros')
print(f'\nDataset listo para limpiar: {df.shape[0]} filas, {df.shape[1]} columnas')

---
## 🔍 Paso 1: Exploración inicial — ¿Qué tenemos?

Antes de limpiar cualquier cosa, necesitamos **entender la estructura del dataset**.

> ⚠️ **Importante:** En este punto `df` ya contiene los problemas inyectados en el Paso 0. Por eso, si miras con atención el output de `df.info()`, ya podrás ver las señales de los problemas:
> - `severity` mostrará **900 non-null** (faltan 100 → 10% nulos inyectados)
> - `inspection_method` mostrará **920 non-null** (faltan 80 → 8% nulos inyectados)
> - `defect_date` mostrará tipo **object** en lugar de datetime (fechas guardadas como texto)
> - `repair_cost` tendrá **1000 non-null** — los costos negativos no son nulos, por eso no se ven aquí; los detectaremos en el Paso 2

En un proyecto real, este paso de exploración es el que alerta al analista de que "algo está raro" antes de entrar al análisis formal de problemas.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPLORACIÓN 1: Ver las primeras filas del dataset
# ════════════════════════════════════════════════════════════════════════

# df.head() muestra las primeras 5 filas del DataFrame.
# Es SIEMPRE el primer paso al trabajar con un dataset nuevo: ver "de qué se trata"
# sin necesidad de abrir el archivo completo (que podría tener millones de filas).
#
# Variaciones útiles:
#   df.head(10) → muestra las primeras 10 filas
#   df.tail(5)  → muestra las últimas 5 filas
#   df.sample(5)→ muestra 5 filas al azar
print('=== Primeras 5 filas del dataset ===')
df.head()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPLORACIÓN 2: Información técnica del dataset
# ════════════════════════════════════════════════════════════════════════

# df.info() muestra un resumen técnico del DataFrame. Para cada columna indica:
#   - Su nombre
#   - Cuántos valores NO nulos tiene (un número menor al total de filas = hay nulos)
#   - El TIPO DE DATO de esa columna. Los tipos más comunes son:
#       int64     → números enteros (sin decimales), ej: 1, 250, -5
#       float64   → números decimales,               ej: 312.5, -0.8
#       object    → texto (cadenas de caracteres),   ej: "Structural", "2023-05-01"
#       datetime64→ fechas y horas (el tipo correcto para fechas)
#       bool      → verdadero o falso (True / False)
#
# Si una columna de "fechas" aparece como "object" → hay un problema de tipo de dato.
# Si "Non-Null Count" < Total entries → hay valores nulos en esa columna.

print('=== Información general del dataset ===')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print()
df.info()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPLORACIÓN 3: Estadísticas descriptivas de las columnas numéricas
# ════════════════════════════════════════════════════════════════════════

# df.describe() calcula automáticamente estadísticas para each columna numérica.
# Solo procesa columnas de tipo int/float — ignora las de texto.
# Las estadísticas que calcula son:
#
#   count → cuántos valores tiene (sin contar nulos)
#   mean  → PROMEDIO: suma de todos los valores ÷ cantidad de valores
#   std   → DESVIACIÓN ESTÁNDAR: qué tan dispersos están los datos alrededor del promedio
#             Un std alto = los datos varían mucho.  Un std bajo = están concentrados.
#   min   → el valor MÁS PEQUEÑO de la columna
#   25%   → PERCENTIL 25: el 25% de los datos está por debajo de este valor
#   50%   → MEDIANA: el valor del medio (50% abajo, 50% arriba)
#   75%   → PERCENTIL 75: el 75% de los datos está por debajo de este valor
#   max   → el valor MÁS GRANDE de la columna
#
# ¿Para qué nos sirve aquí?
#   → Si el "min" de repair_cost es negativo, ya sabemos que hay un problema.
#   → Si "count" es menor al total de filas, hay nulos en esa columna.

print('=== Estadísticas descriptivas (columnas numéricas) ===')
df.describe()

---
## 🚨 Paso 2: Detectar problemas de calidad

Ahora que conocemos la estructura, vamos a **identificar los tres problemas** que sabemos que existen:
1. Valores nulos en `severity` e `inspection_method`
2. Costos de reparación negativos (imposibles)
3. Columna `defect_date` almacenada como texto en vez de fecha

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DETECCIÓN DE PROBLEMA 1: Valores nulos (celdas vacías)
# ════════════════════════════════════════════════════════════════════════

# PASO 1: Contar los nulos por columna
# df.isnull() → crea una tabla idéntica al df pero con True/False:
#   True  = esa celda está VACÍA  (nulo, NaN)
#   False = esa celda TIENE DATO
# .sum() → suma verticalmente (por columna). True cuenta como 1, False como 0.
#   Resultado: cuántas celdas vacías tiene cada columna.
nulos = df.isnull().sum()

# PASO 2: Convertir la cantidad de nulos a porcentaje
# nulos / len(df) → divide cada conteo por el total de filas → fracción (0.0 a 1.0)
# * 100   → multiplica por 100 para pasar a porcentaje (0% a 100%)
# .round(1) → redondea a 1 decimal (ej: 9.979... → 10.0)
porcentaje_nulos = (nulos / len(df) * 100).round(1)

# PASO 3: Combinar ambas métricas en una sola tabla para verlas juntas
# pd.DataFrame({'nombre_col1': datos1, 'nombre_col2': datos2}) crea un nuevo DataFrame
# a partir de un diccionario Python. Un diccionario usa llaves {} y tiene pares clave:valor.
resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})

print('=== Valores nulos por columna ===')
# FILTRO BOOLEANO: resumen_nulos['Nulos'] > 0 genera una serie True/False
# Al usarla dentro de corchetes [...] filtra el DataFrame y muestra
# SOLO las filas donde la condición es True (columnas que tienen al menos 1 nulo).
# Esto equivale al filtro de Excel: "mostrar solo filas donde Nulos > 0"
print(resumen_nulos[resumen_nulos['Nulos'] > 0])
print(f'\nTotal de celdas vacías en todo el dataset: {nulos.sum()}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DETECCIÓN DE PROBLEMA 2: Costos de reparación negativos (imposibles)
# ════════════════════════════════════════════════════════════════════════

# df['repair_cost'] < 0 → compara CADA valor de la columna repair_cost con 0.
# Genera una serie True/False para cada fila:
#   True  = ese costo es negativo (error de datos)
#   False = ese costo es válido (cero o positivo)
#
# df[ condición ] → FILTRO BOOLEANO: retiene SOLO las filas donde la condición es True.
# Es exactamente lo mismo que hacer un filtro en Excel sobre una columna.
# El resultado se guarda en "costos_negativos" (un DataFrame más pequeño).
costos_negativos = df[df['repair_cost'] < 0]

print(f'=== Registros con costo negativo: {len(costos_negativos)} ===')

# Para no mostrar las 8 columnas completas, seleccionamos solo las relevantes.
# df[['col1', 'col2', 'col3', 'col4']] → nota los DOBLES CORCHETES:
#   el corchete exterior es el selector del DataFrame
#   el corchete interior es una LISTA de nombres de columnas
# .head(10) → muestra las primeras 10 filas del conjunto filtrado
print(costos_negativos[['defect_id', 'defect_type', 'severity', 'repair_cost']].head(10))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DETECCIÓN DE PROBLEMA 3: Columna 'defect_date' guardada como texto
# ════════════════════════════════════════════════════════════════════════

# .dtype es una PROPIEDAD (no función, sin paréntesis) que muestra el tipo
# de dato de una columna específica del DataFrame.
# Recuerda: el tipo correcto para fechas es "datetime64[ns]", no "object".
print(f'Tipo de dato de la columna defect_date: {df["defect_date"].dtype}')

# df["defect_date"].head(5) → devuelve las primeras 5 valores de la columna (una Series)
# .tolist() → convierte esa Series de pandas a una lista simple de Python
# Útil para mostrarlo de forma más legible en el print()
print(f'Ejemplos de valores: {df["defect_date"].head(5).tolist()}')
print()

# ¿Cuál es el problema?
# Aunque los valores "parecen" fechas (2023-05-15), pandas los está tratando
# como texto simple. Esto significa que NO podemos:
#   - Calcular cuántos días pasaron entre dos fechas
#   - Filtrar "todos los defectos del mes de marzo"
#   - Agrupar por mes o por año
#   - Ordenar cronológicamente de forma correcta
# Necesitamos CONVERTIR esa columna al tipo datetime (= tipo fecha de pandas).
print('⚠️  La columna es tipo "object" (texto). Debemos convertirla a datetime para poder'
      ' filtrar por fecha y calcular tendencias temporales.')

---
## 🧹 Paso 3: Limpieza de datos

Ahora que sabemos **qué está mal**, vamos a corregirlo sistemáticamente. Este es el corazón del trabajo de un analista de datos.

El orden importa: primero convertimos tipos, luego tratamos nulos, y finalmente eliminamos registros inválidos.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CORRECCIÓN 1: Convertir 'defect_date' de texto (object) → tipo fecha (datetime)
# ════════════════════════════════════════════════════════════════════════

# BUENA PRÁCTICA: Antes de modificar datos, siempre hacer una COPIA del DataFrame original.
#
# df.copy() crea una copia completamente independiente del DataFrame.
# Si escribiéramos solo  df_limpio = df  (sin .copy()),
# ambas variables apuntarían al MISMO objeto en memoria, y cualquier cambio
# en df_limpio también cambiaría df — lo que podría generar errores difíciles de detectar.
# Con .copy(), los cambios en df_limpio NO afectan al df original.
df_limpio = df.copy()

# pd.to_datetime(columna) convierte una columna de texto que contiene fechas
# al tipo datetime64 de pandas. pandas reconoce automáticamente formatos comunes:
#   "2023-05-15" → año-mes-día (ISO 8601)
#   "15/05/2023" → día/mes/año (europeo)
#   "May 15, 2023" → formato inglés
#
# df_limpio['defect_date'] = ... → sobreescribe la columna con los valores convertidos.
# El resultado reemplaza los textos por objetos datetime reales.
df_limpio['defect_date'] = pd.to_datetime(df_limpio['defect_date'])

print(f'✅ Tipo de dato después de la conversión: {df_limpio["defect_date"].dtype}')
# .min() → devuelve el valor mínimo de la columna (la fecha más antigua)
# .max() → devuelve el valor máximo (la fecha más reciente)
# .date() → extrae solo la parte de fecha (año-mes-día), descartando la hora
print(f'   Rango de fechas: {df_limpio["defect_date"].min().date()} → {df_limpio["defect_date"].max().date()}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CORRECCIÓN 2: Rellenar valores nulos en columnas de texto (categóricas)
# ════════════════════════════════════════════════════════════════════════

# Estrategia elegida: reemplazar np.nan con un valor "centinela" (una categoría especial).
#
# Alternativas posibles y sus consecuencias:
#   ✗ Eliminar las filas con nulos  → perdemos información valiosa del resto de columnas
#   ✗ Rellenar con el valor más frecuente (moda) → puede sesgar el análisis
#   ✓ Marcar como "Desconocida"    → mantenemos la fila y quedamos transparentes sobre el dato

# .fillna('texto') reemplaza cada np.nan de la columna por el texto indicado.
# Los valores que ya tienen dato NO se tocan — solo se afectan las celdas vacías.
# df_limpio['severity'] = ... → guardamos el resultado de vuelta en la misma columna
#   (sobreescribimos la columna existente con la versión ya rellenada)
df_limpio['severity']          = df_limpio['severity'].fillna('Desconocida')
df_limpio['inspection_method'] = df_limpio['inspection_method'].fillna('No registrado')

print('✅ Nulos en columnas categóricas reemplazados:')
# .unique() → devuelve un array con todos los valores DISTINTOS que aparecen en la columna
#   Equivale a "Eliminar duplicados" en Excel.
# sorted(...) → ordena esos valores alfabéticamente para mostrarlos más prolijos
print(f'   severity — categorías disponibles:          {sorted(df_limpio["severity"].unique())}')
print(f'   inspection_method — categorías disponibles: {sorted(df_limpio["inspection_method"].unique())}')

# Verificación final: ¿quedan nulos pendientes?
# .isnull().sum().sum() → total de celdas vacías en todo el DataFrame
print(f'\nNulos restantes en el DataFrame: {df_limpio.isnull().sum().sum()}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CORRECCIÓN 3: Eliminar filas con costo de reparación negativo
# ════════════════════════════════════════════════════════════════════════

# Un costo negativo es un error de ingreso de datos. No podemos "corregirlo"
# sin conocer el valor original real. La decisión correcta es eliminar esas filas.

# Guardamos cuántas filas tenemos ANTES del filtro, para reportar cuántas se eliminaron
filas_antes = len(df_limpio)

# FILTRO: df_limpio['repair_cost'] >= 0 → True en filas con costo válido (≥ 0)
# df_limpio[ condición ] → retiene SOLO las filas donde la condición es True
#   Las filas con costos negativos (condición False) quedan fuera automáticamente.
#
# .reset_index(drop=True) → después de filtrar, las filas quedan con índices "saltados"
#   (por ejemplo: 0, 1, 3, 7, 8, ... porque la 2 y la 4-6 se eliminaron).
#   reset_index() renumera de 0 a N-1 de forma continua, como una tabla limpia.
#   drop=True → descarta el índice viejo en vez de agregarlo como columna extra.
df_limpio = df_limpio[df_limpio['repair_cost'] >= 0].reset_index(drop=True)

filas_despues = len(df_limpio)

print(f'✅ Filas eliminadas por costo negativo: {filas_antes - filas_despues}')
print(f'   Filas restantes en el dataset limpio: {filas_despues}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESUMEN DEL PROCESO DE LIMPIEZA
# ════════════════════════════════════════════════════════════════════════
# Las líneas siguientes solo imprimen texto formateado — no hay lógica nueva.
# El formato con ╔ ║ ╚ es solo cosmético (caracteres Unicode de caja).
#
# Nota sobre f-strings y formateo:
#   {len(df):>5}  → imprime len(df) con ancho de 5 caracteres, alineado a la DERECHA
#   Esto hace que los números queden alineados en la tabla de texto.
print('╔══════════════════════════════════════════════════════╗')
print('║          RESUMEN DEL PROCESO DE LIMPIEZA            ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Filas originales (con errores):    {len(df):>5}             ║')
print(f'║  Filas en dataset limpio:           {len(df_limpio):>5}             ║')
print(f'║  Filas eliminadas (costos neg.):    {len(df) - len(df_limpio):>5}             ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Correcciones realizadas:                           ║')
print('║    1. defect_date → convertida a datetime           ║')
print('║    2. severity → nulos → "Desconocida"              ║')
print('║    3. inspection_method → nulos → "No registrado"  ║')
print('║    4. repair_cost < 0 → filas eliminadas            ║')
print('╚══════════════════════════════════════════════════════╝')

# df.dtypes → lista el tipo de dato de CADA columna del DataFrame
# Es buena práctica verificar los tipos finales para confirmar que:
#   - defect_date quedó como datetime64 (no como object)
#   - repair_cost sigue siendo float64
print('\nTipos de dato del DataFrame limpio (verificación final):')
print(df_limpio.dtypes)

---
## 📊 Paso 4: Calcular métricas de negocio

Con los datos limpios, ahora podemos responder la **pregunta de negocio**:

> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

Calcularemos:
1. Distribución de defectos por tipo y por severidad
2. Costo promedio de reparación por tipo de defecto
3. Costo promedio de reparación por severidad
4. Cantidad de defectos detectados por cada método de inspección

---

### 📐 Conceptos clave para esta sección: `value_counts()` y `groupby()`

**`df['columna'].value_counts()`** → cuenta cuántas veces aparece cada valor distinto en una columna.

```python
df['defect_type'].value_counts()
# Resultado:
#   Structural    352
#   Functional    339
#   Cosmetic      309
```

---

**`df.groupby('columna')`** → agrupa todas las filas que comparten el mismo valor en esa columna, y luego aplica una función (promedio, suma, conteo) a cada grupo. Es equivalente a una **tabla dinámica en Excel**.

```python
# Promedio de repair_cost para cada tipo de defecto:
df.groupby('defect_type')['repair_cost'].mean()

# Equivalente en Excel:  tabla dinámica con defect_type en filas y PROMEDIO de repair_cost como valor
```

Visualizando el proceso de groupby:

| defect_type | repair_cost |  →  groupby('defect_type').mean()  →  | defect_type | repair_cost |
|---|---|---|---|---|
| Structural | 800 | | Structural | 525 |
| Cosmetic | 300 | | Functional | 497 |
| Structural | 250 | | Cosmetic | 510 |
| Functional | 497 | | | |

**`.agg(nombre='función')`** → aplica varias funciones a la vez sobre cada grupo, y les da nombres personalizados a las columnas resultantes.

```python
df.groupby('defect_type')['repair_cost'].agg(
    n_defectos   = 'count',   # cuántas filas tiene el grupo
    costo_prom   = 'mean',    # promedio del grupo
    costo_total  = 'sum'      # suma del grupo
)
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 1: Distribución de defectos por TIPO
# ════════════════════════════════════════════════════════════════════════

# df_limpio['defect_type'].value_counts() cuenta cuántas filas hay de cada categoría.
# Por defecto ordena de mayor a menor.
# El resultado es una Series donde: índice = nombre de categoría, valor = conteo
conteo_tipo = df_limpio['defect_type'].value_counts()

# Calculamos el porcentaje de cada tipo respecto al total:
#   conteo_tipo / conteo_tipo.sum() → cada conteo dividido por el total = fracción (0.0-1.0)
#   * 100  → convertir a porcentaje (0%-100%)
#   .round(1) → redondear a 1 decimal
pct_tipo = (conteo_tipo / conteo_tipo.sum() * 100).round(1)

print('=== Distribución por tipo de defecto ===')
# .items() en una Series devuelve pares (índice, valor) para iterar:
#   en cada vuelta del for, "tipo" recibe el nombre (ej: "Structural")
#                           "cnt"  recibe el conteo (ej: 352)
# {tipo:12s} → imprime el texto "tipo" con ancho mínimo de 12 caracteres (alineación)
# {cnt:4d}   → imprime el número "cnt" con ancho mínimo de 4 caracteres (entero)
# pct_tipo[tipo] → busca el porcentaje correspondiente a ese tipo en la serie pct_tipo
for tipo, cnt in conteo_tipo.items():
    print(f'  {tipo:12s}: {cnt:4d} defectos  ({pct_tipo[tipo]}%)')

print()

# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 2: Distribución de defectos por SEVERIDAD
# ════════════════════════════════════════════════════════════════════════
# Mismo proceso que la Métrica 1, pero sobre la columna 'severity'
conteo_sev = df_limpio['severity'].value_counts()
pct_sev    = (conteo_sev / conteo_sev.sum() * 100).round(1)

print('=== Distribución por severidad ===')
for sev, cnt in conteo_sev.items():
    print(f'  {sev:13s}: {cnt:4d} defectos  ({pct_sev[sev]}%)')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 3: Costo promedio de reparación por TIPO DE DEFECTO
# ════════════════════════════════════════════════════════════════════════

# Desglose de la operación encadenada paso a paso:
#
#   df_limpio.groupby('defect_type')
#     → agrupa todas las filas del df según el valor de 'defect_type'
#     → crea 3 grupos: uno para Structural, uno para Functional, uno para Cosmetic
#
#   ['repair_cost']
#     → dentro de cada grupo, trabajamos solo con la columna repair_cost
#
#   .agg(n_defectos='count', costo_promedio='mean', costo_total='sum')
#     → aplica 3 funciones a cada grupo y les da nombres personalizados:
#         n_defectos    = 'count' → cuenta cuántas filas tiene el grupo
#         costo_promedio = 'mean' → promedio de repair_cost en ese grupo
#         costo_total    = 'sum'  → suma total de repair_cost en ese grupo
#
#   .round(2) → redondea todos los resultados a 2 decimales
#
#   .sort_values('costo_promedio', ascending=False)
#     → ordena el resultado por la columna 'costo_promedio' de mayor a menor
#       ascending=False = orden descendente (el más costoso primero)
costo_por_tipo = (
    df_limpio.groupby('defect_type')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean', costo_total='sum')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por tipo de defecto ===')
# .to_string() convierte el DataFrame a texto formateado, mostrando TODAS las filas
# (sin truncar, a diferencia del display normal)
print(costo_por_tipo.to_string())

# .idxmax() → devuelve el NOMBRE del índice (aquí, el tipo de defecto) con el valor máximo
# .max()    → devuelve el VALOR máximo (el costo promedio más alto)
print(f'\n➡  El tipo más costoso es: {costo_por_tipo["costo_promedio"].idxmax()}'
      f'  (${costo_por_tipo["costo_promedio"].max():.2f} promedio)')

print()

# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 4: Costo promedio de reparación por SEVERIDAD
# ════════════════════════════════════════════════════════════════════════

# Antes de agrupar, filtramos los registros con severidad "Desconocida"
# para no contaminar el análisis con datos sin información real.
# df_limpio[df_limpio['severity'] != 'Desconocida']
#   → != significa "diferente de"
#   → retiene solo filas donde severity es Minor, Moderate o Critical
costo_por_sev = (
    df_limpio[df_limpio['severity'] != 'Desconocida']
    .groupby('severity')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por severidad ===')
print(costo_por_sev.to_string())

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 5: ¿Cuántos defectos detecta cada método de inspección?
# ════════════════════════════════════════════════════════════════════════

# Excluimos registros con inspection_method = 'No registrado' para no distorsionar
# el análisis — esa "categoría" fue creada por nosotros para reemplazar nulos.
# Si la incluimos, crearíamos una categoría artificial que no representa un método real.
df_con_metodo = df_limpio[df_limpio['inspection_method'] != 'No registrado']

# .groupby('inspection_method') → agrupa por método de inspección
# .agg con sintaxis extendida: nombre_col=('columna_fuente', 'función')
#   n_defectos_detectados    = cuántos defect_id hay en cada grupo (= cuántos defectos detectó)
#   costo_promedio_detectado = promedio de repair_cost de defectos que detectó cada método
efectividad = (
    df_con_metodo.groupby('inspection_method')
    .agg(
        n_defectos_detectados   = ('defect_id',   'count'),
        costo_promedio_detectado = ('repair_cost', 'mean')
    )
    .round(2)
    .sort_values('n_defectos_detectados', ascending=False)
)

print('=== Métodos de inspección — defectos detectados ===')
print(efectividad.to_string())

print()

# ════════════════════════════════════════════════════════════════════════
# MÉTRICA 6: ¿Qué método detecta más defectos CRÍTICOS específicamente?
# ════════════════════════════════════════════════════════════════════════

# Encadenamos DOS filtros antes de agrupar:
#   Filtro 1: df_con_metodo[...] → solo filas con método registrado (ya aplicado arriba)
#   Filtro 2: df_con_metodo['severity'] == 'Critical' → además, solo defectos críticos
#   == significa "igual a" (doble signo igual → comparación, no asignación)
#
# Luego agrupamos y contamos cuántos defect_id hay en cada grupo (= cuántos críticos detectó)
criticos_por_metodo = (
    df_con_metodo[df_con_metodo['severity'] == 'Critical']
    .groupby('inspection_method')['defect_id']
    .count()
    .sort_values(ascending=False)
)

print('=== Defectos CRÍTICOS detectados por método de inspección ===')
print(criticos_por_metodo.to_string())
print(f'\n➡  El método más efectivo para defectos críticos: {criticos_por_metodo.idxmax()}'
      f'  ({criticos_por_metodo.max()} defectos críticos)')

---
## 📈 Paso 5: Visualizaciones

Cinco gráficos para presentar los hallazgos a gerencia:
1. **Defectos por tipo** — distribución relativa por `defect_type`
2. **Costo promedio por severidad** — ¿los defectos críticos cuestan más?
3. **Tendencia temporal** — ¿hay meses con picos de defectos?
4. **Costo promedio por tipo de defecto** — responde directamente la primera parte de la pregunta de negocio
5. **Efectividad de los métodos de inspección** — responde la segunda parte: ¿qué método detecta más defectos, y más defectos críticos?


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GRÁFICO 1: Cantidad de defectos por tipo de defecto
# GRÁFICO 2: Costo promedio de reparación por severidad
# (ambos gráficos se muestran lado a lado en la misma figura)
# ════════════════════════════════════════════════════════════════════════

# plt.subplots(filas, columnas, figsize=(ancho, alto)) crea un "lienzo" con múltiples zonas.
#   filas=1, columnas=2 → una fila, dos gráficos lado a lado
#   figsize=(13, 5)     → tamaño del lienzo en pulgadas (ancho=13, alto=5)
# Devuelve dos variables:
#   fig   → la figura completa (el "lienzo" que contiene todo)
#   axes  → array con los dos "ejes" (zonas de dibujo): axes[0]=izquierda, axes[1]=derecha
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# fig.suptitle() pone un título general que abarca TODA la figura (sobre los dos gráficos)
# fontsize=14 → tamaño de fuente 14pt
# fontweight='bold' → texto en negrita
# y=1.01 → posición vertical del título (1.01 = ligeramente por encima del borde superior)
fig.suptitle('Análisis de Defectos de Manufactura', fontsize=14, fontweight='bold', y=1.01)

# ────────────────────────────────────────────────────────────────────────
# GRÁFICO 1 (zona izquierda, axes[0])
# ────────────────────────────────────────────────────────────────────────
ax1 = axes[0]   # Asignamos el primer eje a la variable ax1 para escribir menos

# Extraemos las categorías y sus conteos de la Series conteo_tipo
# .index.tolist() → convierte el índice de la Series (nombres de tipo) a lista Python
# .values.tolist() → convierte los valores (cantidades) a lista Python
tipos  = conteo_tipo.index.tolist()
counts = conteo_tipo.values.tolist()

# Colores en código hexadecimal: #RRGGBB (Red, Green, Blue en base 16)
# #2196F3 = azul Material Design, #FF9800 = naranja, #4CAF50 = verde
colores_tipo = ['#2196F3', '#FF9800', '#4CAF50']

# ax1.bar(x, alturas, ...) dibuja barras verticales.
#   x = tipos        → las etiquetas del eje horizontal (nombres de columnas)
#   counts           → la altura de cada barra
#   color            → color de relleno de cada barra
#   edgecolor='white'→ borde blanco entre barras (efecto visual más limpio)
#   linewidth=1.2    → grosor del borde
# Devuelve "bars1": la colección de barras dibujadas (la usamos después para añadir etiquetas)
bars1 = ax1.bar(tipos, counts, color=colores_tipo, edgecolor='white', linewidth=1.2)

# Añadimos etiquetas de número encima de cada barra (para que el lector vea el valor exacto)
for bar in bars1:
    # bar.get_x()      → posición del borde izquierdo de la barra
    # bar.get_width()  → ancho de la barra
    # get_x() + get_width()/2  → posición del CENTRO horizontal de la barra
    # bar.get_height() → altura de la barra (= el conteo)
    # bar.get_height() + 5 → un poco por ENCIMA del tope de la barra
    # str(int(...))    → convierte la altura (float) a entero y luego a texto
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 5,
             str(int(bar.get_height())),
             ha='center',     # alineación horizontal = centrado
             va='bottom',     # alineación vertical = la base del texto en la posición indicada
             fontsize=11, fontweight='bold')

ax1.set_title('Cantidad de Defectos por Tipo', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tipo de defecto')     # Etiqueta del eje X (horizontal)
ax1.set_ylabel('Cantidad de defectos') # Etiqueta del eje Y (vertical)
ax1.set_ylim(0, max(counts) * 1.15)  # Límite del eje Y: 15% más arriba que la barra más alta
                                      # → dejar espacio para las etiquetas de número
ax1.grid(axis='y', linestyle='--', alpha=0.4) # Líneas horizontales de referencia
                                               # linestyle='--' = línea punteada
                                               # alpha=0.4 = 40% de opacidad (suave)
ax1.spines[['top', 'right']].set_visible(False) # Ocultar bordes superior y derecho
                                                 # → estilo más limpio y moderno

# ────────────────────────────────────────────────────────────────────────
# GRÁFICO 2 (zona derecha, axes[1])
# ────────────────────────────────────────────────────────────────────────
ax2 = axes[1]

# Definimos el orden que queremos en el eje X (menor a mayor severidad)
sev_orden = ['Minor', 'Moderate', 'Critical']

# Para cada severidad en ese orden, buscamos su costo promedio en costo_por_sev.
# Usamos una LIST COMPREHENSION (forma compacta de crear una lista con un for + condición):
#   [expresión   for variable in lista]
# costo_por_sev.loc[s, 'costo_promedio'] → accede al valor de la fila "s" en esa columna
# El 'if ... else 0' es un valor de respaldo por si alguna categoría no está en el índice
sev_costos = [
    costo_por_sev.loc[s, 'costo_promedio'] if s in costo_por_sev.index else 0
    for s in sev_orden
]

colores_sev = ['#4CAF50', '#FF9800', '#F44336']  # verde, naranja, rojo

bars2 = ax2.bar(sev_orden, sev_costos, color=colores_sev, edgecolor='white', linewidth=1.2)

# zip(lista1, lista2) combina dos listas en pares: (elem1a, elem1b), (elem2a, elem2b)...
# Nos permite iterar sobre las barras y sus valores al mismo tiempo
for bar, val in zip(bars2, sev_costos):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 3,
             f'${val:.0f}',   # {:.0f} → número sin decimales con el prefijo $
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_title('Costo Promedio de Reparación por Severidad', fontsize=12, fontweight='bold')
ax2.set_xlabel('Severidad')
ax2.set_ylabel('Costo promedio ($)')
ax2.set_ylim(0, max(sev_costos) * 1.15)
ax2.grid(axis='y', linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

# plt.tight_layout() ajusta automáticamente los márgenes para que nada se superponga
plt.tight_layout()

# plt.savefig() guarda la figura como archivo de imagen en la misma carpeta del notebook
# dpi=150     → resolución (150 puntos por pulgada, buena calidad)
# bbox_inches='tight' → no recortar nada al guardar
plt.savefig('grafico_defectos_analisis.png', dpi=150, bbox_inches='tight')

# plt.show() muestra el gráfico en pantalla (en Colab aparece debajo de esta celda)
plt.show()
print('✅ Gráfico guardado como: grafico_defectos_analisis.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GRÁFICO 3: Tendencia mensual de defectos detectados (gráfico de línea)
# ════════════════════════════════════════════════════════════════════════

# Para agrupar por mes necesitamos una columna que contenga solo el mes y año,
# sin el día. Usamos el "accessor" .dt de pandas, que permite acceder a partes de fechas.
#
# .dt.to_period('M') convierte cada fecha exacta a un "período mensual":
#   datetime 2023-05-15  →  Period '2023-05'  (solo mes y año, sin el día)
#   datetime 2023-05-28  →  Period '2023-05'  ← mismo período → misma barra
#   datetime 2023-06-03  →  Period '2023-06'  ← período siguiente
#
# Guardamos el resultado en una NUEVA columna llamada 'mes' del DataFrame
df_limpio['mes'] = df_limpio['defect_date'].dt.to_period('M')

# Agrupamos por mes y contamos cuántos defectos hubo en cada período
# .groupby('mes')['defect_id'].count()
#   → para cada mes, cuenta cuántos defect_id hay = cuántos defectos se registraron
# Resultado: una Series con índice=mes (Period), valor=cantidad de defectos
tendencia_mensual = df_limpio.groupby('mes')['defect_id'].count()

# plt.subplots(figsize=...) sin argumentos de filas/columnas → 1 solo gráfico
fig, ax = plt.subplots(figsize=(12, 4))

# ax.plot() dibuja una línea que conecta todos los puntos de datos
# .index.astype(str) → convierte los Period('2023-05') a texto '2023-05' para el eje X
#   (matplotlib necesita texto o números para los ejes, no objetos Period)
# .values → los valores numéricos (cantidad de defectos) para el eje Y
# marker='o'    → dibuja un punto circular en cada dato
# linewidth=2   → grosor de la línea
# color='#1565C0' → azul oscuro
# markersize=5  → tamaño de los puntos circulares
ax.plot(tendencia_mensual.index.astype(str),
        tendencia_mensual.values,
        marker='o', linewidth=2, color='#1565C0', markersize=5)

# ax.fill_between() rellena el área entre la línea y el eje X
# Crea un efecto de "área sombreada" que hace más visible la tendencia
# alpha=0.15 → 15% de opacidad (muy transparente para no opacar la línea)
ax.fill_between(tendencia_mensual.index.astype(str),
                tendencia_mensual.values,
                alpha=0.15, color='#1565C0')

ax.set_title('Tendencia Mensual de Defectos Detectados', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Cantidad de defectos')

# rotation=45 → rota las etiquetas del eje X 45° para que no se solapen entre sí
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# Si el rango de fechas cubre muchos meses, el eje X se satura de etiquetas.
# Solución: mostrar solo 1 de cada 2 etiquetas.
# enumerate(lista) devuelve pares (índice_posición, elemento):
#   (0, '2023-01'), (1, '2023-02'), (2, '2023-03'), ...
# i % 2 → operador módulo (resto de la división): si i es par → 0, si es impar → 1
# Si i % 2 != 0 (posición impar) → ocultamos esa etiqueta
for i, label in enumerate(ax.get_xticklabels()):
    if i % 2 != 0:
        label.set_visible(False)

plt.tight_layout()
plt.savefig('grafico_tendencia_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_tendencia_mensual.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GRÁFICO 4: Costo promedio de reparación por TIPO DE DEFECTO
# GRÁFICO 5: Defectos detectados por MÉTODO DE INSPECCIÓN
# (responden directamente la pregunta de negocio)
# ════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Respuesta a la Pregunta de Negocio', fontsize=14, fontweight='bold', y=1.01)

# ────────────────────────────────────────────────────────────────────────
# GRÁFICO 4 (izquierda): Costo promedio por tipo de defecto
# Responde: "¿Qué tipo de defecto genera mayor costo de reparación?"
# ────────────────────────────────────────────────────────────────────────
ax4 = axes[0]

# costo_por_tipo fue calculado en el Paso 4 con groupby + agg
# Sus filas ya están ordenadas de mayor a menor costo_promedio
tipos_g4  = costo_por_tipo.index.tolist()
costos_g4 = costo_por_tipo['costo_promedio'].tolist()

# Colorear la barra más alta (índice 0 al estar ya ordenado desc) en naranja acento
colores_g4 = ['#EA7600' if i == 0 else '#90CAF9' for i in range(len(tipos_g4))]

bars4 = ax4.bar(tipos_g4, costos_g4, color=colores_g4, edgecolor='white', linewidth=1.2)

# Etiquetas de valor encima de cada barra
for bar, val in zip(bars4, costos_g4):
    ax4.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 3,
             f'${val:.0f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax4.set_title('Costo Promedio de Reparación por Tipo\n(barra naranja = el más costoso)',
              fontsize=11, fontweight='bold')
ax4.set_xlabel('Tipo de defecto')
ax4.set_ylabel('Costo promedio ($)')
ax4.set_ylim(0, max(costos_g4) * 1.20)
ax4.grid(axis='y', linestyle='--', alpha=0.4)
ax4.spines[['top', 'right']].set_visible(False)

# ────────────────────────────────────────────────────────────────────────
# GRÁFICO 5 (derecha): Defectos detectados por método de inspección
# Responde: "¿Qué método de inspección es más efectivo para detectarlo?"
# Mostramos barras agrupadas: total de defectos y defectos críticos por método
# ────────────────────────────────────────────────────────────────────────
ax5 = axes[1]

# Ordenamos los métodos igual que en 'efectividad' (por n_defectos desc)
metodos = efectividad.index.tolist()

# Total de defectos detectados por cada método
total_detectados = efectividad['n_defectos_detectados'].tolist()

# Críticos detectados por cada método (de la Serie criticos_por_metodo)
# .get(m, 0) devuelve 0 si el método no aparece en la Serie (valor de respaldo seguro)
criticos = [criticos_por_metodo.get(m, 0) for m in metodos]

# Barras agrupadas: necesitamos posiciones X manuales para que quepan 2 barras por grupo
import numpy as np
x = np.arange(len(metodos))   # [0, 1, 2] — posición central de cada grupo
ancho = 0.35                   # ancho de cada barra individual

# plot.bar() acepta un array de posiciones X; ax5.bar(x - ancho/2) desplaza las barras a la izquierda
bars_total    = ax5.bar(x - ancho / 2, total_detectados, ancho,
                        label='Total detectados', color='#42A5F5', edgecolor='white')
bars_criticos = ax5.bar(x + ancho / 2, criticos, ancho,
                        label='Defectos críticos', color='#EF5350', edgecolor='white')

# Etiquetas encima de cada barra
for bar in bars_total:
    ax5.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in bars_criticos:
    ax5.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='#C62828')

# Etiquetas del eje X: los nombres de los métodos centrados en cada grupo
ax5.set_xticks(x)
ax5.set_xticklabels(metodos, rotation=12, ha='right')

ax5.set_title('Defectos Detectados por Método de Inspección\n(azul = total, rojo = críticos)',
              fontsize=11, fontweight='bold')
ax5.set_xlabel('Método de inspección')
ax5.set_ylabel('Cantidad de defectos')
ax5.set_ylim(0, max(total_detectados) * 1.20)
ax5.legend(fontsize=9)
ax5.grid(axis='y', linestyle='--', alpha=0.4)
ax5.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_pregunta_negocio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_pregunta_negocio.png')
print()
print('=== Respuesta a la pregunta de negocio ===')
print(f'  Tipo más costoso    : {costo_por_tipo["costo_promedio"].idxmax()}'
      f'  (${costo_por_tipo["costo_promedio"].max():.2f} promedio)')
print(f'  Método más efectivo (total)    : {efectividad["n_defectos_detectados"].idxmax()}'
      f'  ({efectividad["n_defectos_detectados"].max()} defectos detectados)')
print(f'  Método más efectivo (críticos) : {criticos_por_metodo.idxmax()}'
      f'  ({criticos_por_metodo.max()} defectos críticos)')


---
## 💾 Paso 6: Exportar los datos limpios

El último paso es guardar el dataset limpio y con la nueva métrica calculada. Este archivo es el que se entregaría al equipo de gerencia, al área de calidad, o que se usaría como entrada para un modelo de IA.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PASO 6: EXPORTAR EL DATASET LIMPIO A UN ARCHIVO CSV
# ════════════════════════════════════════════════════════════════════════

# Guardamos el nombre del archivo en una variable para reutilizarlo
nombre_archivo = 'defects_data_limpio.csv'

# df_limpio.to_csv('nombre.csv', index=False) guarda el DataFrame como archivo CSV.
#
# ¿Qué hace to_csv?
#   1. Crea un archivo de texto nuevo en la misma carpeta del notebook
#   2. Escribe los nombres de las columnas en la primera línea
#   3. Escribe cada fila del DataFrame como una línea delimitada por comas
#
# index=False → MUY IMPORTANTE: no incluir la columna de índice (0, 1, 2, 3, ...)
#   como una columna extra al inicio del CSV.
#   Si usáramos index=True (o no lo indicáramos), el CSV tendría una columna llamada
#   "Unnamed: 0" con los números de fila, lo cual suele ser indeseable.
df_limpio.to_csv(nombre_archivo, index=False)

print(f'✅ Archivo exportado: {nombre_archivo}')
print(f'   Filas exportadas: {len(df_limpio)}')
# list(df_limpio.columns) → convierte el índice de columnas a una lista Python simple
print(f'   Columnas incluidas: {list(df_limpio.columns)}')
print()
print('Vista previa del archivo exportado (primeras 5 filas):')
# df_limpio.head() muestra las primeras 5 filas del DataFrame limpio
# Es la última verificación: confirmar que el archivo exportado tiene los datos correctos
df_limpio.head()

---
## ✅ Conclusiones

### Respuesta a la pregunta de negocio
> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

A partir del análisis del dataset de defectos de manufactura (1.000 registros), los hallazgos clave son:

1. **Tipo de defecto más costoso:** Los tres tipos (Structural, Functional, Cosmetic) tienen costos promedio similares (~$500), lo que sugiere que **el tipo no es el principal predictor del costo** — la severidad sí lo es.

2. **Severidad y costo:** Los defectos **Critical** son consistentemente más costosos que los Minor o Moderate. Esto tiene sentido operacional: los defectos críticos requieren más trabajo de reparación o reemplazo de piezas.

3. **Método de inspección:** Los tres métodos (Manual Testing, Visual Inspection, Automated Testing) detectan volúmenes similares de defectos. Sin embargo, para defectos críticos, los métodos automatizados y manuales tienden a superar a la inspección visual — sugiriendo que la **inspección visual puede pasar por alto defectos internos o estructurales**.

---

### Lo que aprendimos sobre calidad de datos
- Un CSV puede llegar con errores de tipo (texto en vez de fechas), valores faltantes y errores de signo
- Los problemas de calidad **no se detectan con el ojo a simple vista** — requieren código sistemático
- La limpieza de datos es el paso que más tiempo consume en un proyecto real de análisis (60–80% del tiempo total)
- Un dato mal guardado puede llevar a conclusiones erróneas y decisiones de negocio incorrectas

---

### Próximos pasos sugeridos
- Agregar visualizaciones por `defect_location` (¿dónde ocurren los defectos?)
- Analizar si hay correlación entre `defect_date` y picos de producción
- Cruzar este dataset con datos de producción para calcular la **tasa de defectos por lote**